In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
df = pd.read_excel(r"Signet Volume.xlsx", engine="openpyxl")

df['Month'] = pd.to_datetime(df['Month'])
df = df.sort_values(['Serial Number', 'Month'])

df = df.groupby(['Serial Number', 'Month'], as_index=False)['Month Volume'].sum()

In [ ]:
def seasonal_naive_forecast(sub, periods, m=12):
    """
    Robust seasonal naive:
    - Works even if len(sub) < m (uses what's available).
    - Handles all-NaN by returning zeros.
    """
    # Use only observed (non-NaN) values for the last season
    sub_nonan = sub.dropna()

    idx = pd.date_range(
        start=sub.index[-1] + pd.offsets.MonthBegin(1),
        periods=periods, freq='MS'
    )

    if len(sub_nonan) == 0:
        # Nothing to go on ➜ forecast zeros
        return pd.Series(np.zeros(periods), index=idx)

    k = min(m, len(sub_nonan))  # effective season length
    last_season = sub_nonan.iloc[-k:].values

    # Tile based on k (actual available last season), not m
    reps = int(np.ceil(periods / k))
    vals = np.tile(last_season, reps)[:periods]

    return pd.Series(vals, index=idx)


def seasonal_average_forecast(sub, periods, m=12, min_years=2):
    """
    Seasonal average by month-of-year.
    Falls back to seasonal naive if:
      - Not enough history per month-of-year, or
      - Too few months (< m), or
      - Missing month means.
    """
    idx = pd.date_range(
        start=sub.index[-1] + pd.offsets.MonthBegin(1),
        periods=periods, freq='MS'
    )

    sub_nonan = sub.dropna()

    # If short history, fallback to seasonal naive
    if len(sub_nonan) < m:
        return seasonal_naive_forecast(sub, periods, m=m)

    df = sub_nonan.to_frame('y').assign(moy=sub_nonan.index.month)
    counts = df.groupby('moy')['y'].count()

    # Ensure at least min_years observations for each month-of-year we’ll need
    if (counts < min_years).any() or counts.size < 12:
        return seasonal_naive_forecast(sub, periods, m=m)

    seasonal_means = df.groupby('moy')['y'].mean()

    future_months = [(sub.index[-1] + pd.offsets.MonthBegin(k)).month for k in range(1, periods+1)]

    # If any future month is missing in seasonal_means for some reason, fallback
    if not set(future_months).issubset(set(seasonal_means.index)):
        return seasonal_naive_forecast(sub, periods, m=m)

    vals = [seasonal_means[moy] for moy in future_months]
    return pd.Series(vals, index=idx)


def hw_forecast(sub, periods, seasonal='add', damped=True, m=12):
    model = ExponentialSmoothing(
        sub, trend='add', damped_trend=damped,
        seasonal=seasonal, seasonal_periods=m,
        initialization_method='estimated'
    )
    fitted = model.fit(optimized=True)
    return fitted.forecast(periods)

def stlf_forecast(sub, periods, m=12, robust=True):
    stlf = STLForecast(sub, ARIMA, model_kwargs={'order': (1,1,1)}, period=m, robust=robust)
    res = stlf.fit()
    return res.forecast(periods)

def sarima_auto_small(sub, periods, m=12):
    import itertools
    pdq = [(p,d,q) for p in [0,1] for d in [0,1] for q in [0,1]]
    PDQ = [(P,D,Q,m) for P in [0,1] for D in [0,1] for Q in [0,1]]
    best_aic, best_res = np.inf, None
    for order in pdq:
        for seasonal_order in PDQ:
            try:
                model = SARIMAX(sub, order=order, seasonal_order=seasonal_order,
                                enforce_stationarity=False, enforce_invertibility=False)
                res = model.fit(disp=False)
                if res.aic < best_aic:
                    best_aic, best_res = res.aic, res
            except Exception:
                pass
    if best_res is None:
        return seasonal_naive_forecast(sub, periods, m=m)
    return best_res.forecast(periods)

def forecast_serial(
    data, periods=6, model_type="stlf", seasonal_periods=12,
    min_history=18, clip_negative=True
):
    results = {}
    serials = data['Serial Number'].unique()

    for serial in serials:
        sub = data[data['Serial Number'] == serial].copy()

        # Ensure datetime & ordering
        sub['Month'] = pd.to_datetime(sub['Month'])
        sub = sub.sort_values('Month').set_index('Month')['Month Volume']

        # Regularize monthly frequency
        sub = sub.asfreq('MS')

        # Prefer not to fill NaNs with zeros:
        # Option 1: leave NaN for SARIMAX/STLF (Kalman filter handles NaN)
        # Option 2 (if not using SARIMAX/STLF): interpolate
        if model_type in ("seasonal_naive", "seasonal_avg", "hw_add", "hw_mul"):
            sub = sub.interpolate('linear')

        # Insufficient history? Use simple seasonal average or naive
        if len(sub.dropna()) < min_history:
            forecast = seasonal_naive_forecast(sub.fillna(method='ffill').fillna(0), periods, m=seasonal_periods)
        else:
            try:
                if model_type == "seasonal_naive":
                    forecast = seasonal_naive_forecast(sub, periods, m=seasonal_periods)
                elif model_type == "seasonal_avg":
                    forecast = seasonal_average_forecast(sub, periods, m=seasonal_periods)
                elif model_type == "hw_add":
                    forecast = hw_forecast(sub, periods, seasonal='add', m=seasonal_periods)
                elif model_type == "hw_mul":
                    forecast = hw_forecast(sub, periods, seasonal='mul', m=seasonal_periods)
                elif model_type == "stlf":
                    forecast = stlf_forecast(sub, periods, m=seasonal_periods)
                elif model_type == "sarima_auto":
                    forecast = sarima_auto_small(sub, periods, m=seasonal_periods)
                else:
                    forecast = seasonal_naive_forecast(sub, periods, m=seasonal_periods)
            except Exception:
                # robust fallback
                forecast = seasonal_naive_forecast(sub.fillna(method='ffill').fillna(0), periods, m=seasonal_periods)

        if clip_negative:
            forecast = forecast.clip(lower=0)

        results[serial] = {"history": sub, "forecast": forecast}

    return results

# Usage:
results = forecast_serial(df, periods=6, model_type="stlf")  # or "hw_add" / "sarima_auto" / "seasonal_avg"
# Build final_df as you already do.


# 4. Convert results to DataFrame
all_data = []
for serial, data in results.items():
    history_df = pd.DataFrame({
        'Serial Number': serial,
        'Month': data['history'].index,
        'Volume': data['history'].values,
        'Type': 'History'
    })
    forecast_df = pd.DataFrame({
        'Serial Number': serial,
        'Month': data['forecast'].index,
        'Volume': data['forecast'].values,
        'Type': 'Forecast'
    })
    combined = pd.concat([history_df, forecast_df])
    all_data.append(combined)

final_df = pd.concat(all_data).reset_index(drop=True)

C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_24784\41544184.py:125: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  forecast = seasonal_naive_forecast(sub.fillna(method='ffill').fillna(0), periods, m=seasonal_periods)
C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_24784\41544184.py:125: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  forecast = seasonal_naive_forecast(sub.fillna(method='ffill').fillna(0), periods, m=seasonal_periods)
C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_24784\41544184.py:125: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  forecast = seasonal_naive_forecast(sub.fillna(method='ffill').fillna(0), periods, m=seasonal_periods)
C:\Users\JSpradlin\AppData\Local\Temp\ipykernel_24784\41544184.py:125: Futur

In [ ]:
# Cleaning out the negative volume serial numbers

drop_sn = final_df[final_df['Volume']<0]['Serial Number'].unique().tolist()
final_df = final_df[~final_df['Serial Number'].isin(drop_sn)]

In [ ]:
# Looking at the count of the historical volumes
final_df[final_df['Type']=='History']['Serial Number'].value_counts()

Serial Number
U64988K1N852892    42
U64988L1N989916    42
U64988L1N989889    42
U64988L1N989894    42
U64988L1N989895    42
                   ..
VND4M11209          1
72HT2BX             1
CNDRP5L114          1
U64221B2N81         1
3400100109NFL       1
Name: count, Length: 4439, dtype: int64

In [ ]:
sn_value = 'U64988K1N852892'

fig = px.line(final_df[final_df['Serial Number']==f'{sn_value}'], 
              x = 'Month',
              y = 'Volume',
              color = 'Type',
              markers = 'circle',
              title = f'{sn_value}',
              template = 'plotly_dark')

fig.show()